# Phase 1: Discontinuation Risk Classification

**Question:** among drugs already on the FDA shortage list, which features predict whether the entry is a permanent discontinuation ("To Be Discontinued") rather than an active/temporary shortage ("Current")?

**Scope note:** this is *not* a "will this drug go into shortage" model — there's no control group of non-shortage drugs in this dataset. It's narrower: given a drug is already listed, what makes a permanent supply loss more likely.

**A methodology note up front**, because it changes how the results below should be read. An early version of this model included `days_since_posted` (how long the entry has sat on the shortage list) and hit AUC 1.000 — a red flag, not a result. That feature is entangled with the label almost by definition (FDA's own bookkeeping tends to close out "To Be Discontinued" entries faster than open-ended "Current" ones), so a model "predicting" discontinuation from it is reading the FDA's paperwork clock, not finding a supply-chain risk factor. It's excluded from the model below.

With it removed, a plain random train/test split *still* gave AUC ~0.986 on categorical features alone. That also needed a second look — see the combo-purity section below — and the honest, generalization-aware estimate turns out to be lower (~0.90-0.93) but still clearly informative.

In [1]:
import sys
sys.path.insert(0, "src")

from shortage_risk import (
    build_dataset, make_pipeline, evaluate, evaluate_grouped,
    grouped_auc_summary, combo_purity, top_features,
    discontinuation_rate_by, MODEL_FEATURES, CAT_FEATURES,
)

df = build_dataset()
print(f"{len(df)} rows, {df['is_discontinuation'].mean():.1%} discontinuation rate")
df.head()

1596 rows, 27.8% discontinuation rate


,generic_name,brand_name,manufacturer_name,company_name,product_ndc,package_ndc,route,dosage_form,therapeutic_category,product_type,...,initial_posting_date,update_date,discontinued_date,related_info,presentation,days_since_posted,is_discontinuation,is_resolved,suppliers_in_shortage_list,combo
0,Sodium Chloride 0.9% Injection,SODIUM CHLORIDE,"Hospira, Inc.","Hospira, Inc., a Pfizer Company",0409-7101,0409-7101-66,INTRAVENOUS,Injection,Gastroenterology,HUMAN PRESCRIPTION DRUG,...,2025-12-03,2025-12-03,2025-12-03,Discontinuation of the manufacture of the drug,"Sodium Chloride 0.9%, Injection, 50 mL ADD-Va...",293,1,0,1,INTRAVENOUS|Injection|Gastroenterology
1,"Montelukast Sodium Tablet, Chewable",SINGULAIR,Organon LLC,Organon,78206-170,78206-170-01,ORAL,"Tablet, Chewable",Pulmonary/Allergy,HUMAN PRESCRIPTION DRUG,...,2025-11-17,2025-11-17,2025-11-17,Discontinuation of the manufacture of the drug,"Singulair, Tablet, Chewable, 4 mg (NDC 78206-1...",309,1,0,1,"ORAL|Tablet, Chewable|Pulmonary/Allergy"
2,Haloperidol Decanoate Injection,HALOPERIDOL DECANOATE,"Fresenius Kabi USA, LLC","Fresenius Kabi USA, LLC",63323-469,63323-469-05,INTRAMUSCULAR,Injection,Psychiatry,HUMAN PRESCRIPTION DRUG,...,2026-01-09,2026-01-09,2026-01-09,Discontinuation of the manufacture of the drug,"Haloperidol Decanoate, Injection, 50 mg/1 mL (...",256,1,0,1,INTRAMUSCULAR|Injection|Psychiatry
3,"Doxycycline Hyclate Tablet, Delayed Release",DOXYCYCLINE HYCLATE,"Actavis Pharma, Inc.","Teva Pharmaceuticals USA, Inc.",0591-4550,0591-4550-22,ORAL,Tablet,Anti-Infective,HUMAN PRESCRIPTION DRUG,...,2026-07-30,2026-07-30,2026-07-30,Discontinuation of the manufacture of the drug,"Doxycycline Hyclate, Tablet, Delayed Release, ...",54,1,0,1,ORAL|Tablet|Anti-Infective
4,Lisdexamfetamine Dimesylate Capsule,LISDEXAMFETAMINE DIMESYLATE,Mylan Pharmaceuticals Inc.,"Mylan Pharmaceuticals Inc., a Viatris Company",0378-6860,0378-6860-77,ORAL,Capsule,Psychiatry,HUMAN PRESCRIPTION DRUG,...,2023-07-14,2026-09-17,NaT,NaN,"Lisdexamfetamine Dimesylate, Capsule, 70 mg (N...",1166,0,0,13,ORAL|Capsule|Psychiatry


## Naive random split (for comparison only)

In [2]:
pipe, naive_auc, report, _ = evaluate(df, model="gbm")
print(f"GBM test AUC (naive random row split): {naive_auc:.3f}")
print(report)

GBM test AUC (naive random row split): 0.986
              precision    recall  f1-score   support

           0       0.95      0.98      0.97       288
           1       0.95      0.87      0.91       111

    accuracy                           0.95       399
   macro avg       0.95      0.93      0.94       399
weighted avg       0.95      0.95      0.95       399



## Why the naive split is inflated: combination overlap

There are only ~140 unique combinations of `route` x `dosage_form` x `therapeutic_category` across ~1,600 rows. A random row split lets the same combination land in both train and test, so part of the score above is the model recognizing a combination it already saw during training, not generalizing to a new drug profile.

In [3]:
purity = combo_purity(df)
for k, v in purity.items():
    print(f"{k}: {v}")

unique_combos: 140
pure_combos: 112
mixed_combos: 28
rows_in_pure_combos: 725
total_rows: 1596


## Honest estimate: combo-grouped split

Splitting by *combination* instead of by row means every combination in the test fold is unseen during training — this is the real test of whether these categorical fields generalize. Run across 5 different random combo splits, since any single split's score depends a lot on which combinations happen to land in the test fold.

In [4]:
summary = grouped_auc_summary(df, n_splits=5)
print(summary.to_string())
print(f"\nmean: {summary.mean():.3f}   std: {summary.std():.3f}")

0    0.838954
1    0.963071
2    0.977840
3    0.963223
4    0.898241

mean: 0.928   std: 0.059


**Result:** mean AUC ~0.93 (std ~0.06) on combinations the model never saw during training. Lower than the naive 0.986, but still a strong, genuine signal — this is the number to report, not the naive one.

## What's actually driving the signal

In [5]:
full_pipe = make_pipeline("gbm")
full_pipe.fit(df[MODEL_FEATURES], df["is_discontinuation"])
top_features(full_pipe, n=10)

suppliers_in_shortage_list             0.795225
dosage_form_Tablet                     0.036528
route_Unknown                          0.026159
therapeutic_category_Oncology          0.021294
dosage_form_Solution                   0.018873
therapeutic_category_Transplant        0.014503
therapeutic_category_Anesthesia        0.013413
dosage_form_Film, Extended Release     0.009920
therapeutic_category_Anti-Infective    0.009781
therapeutic_category_Cardiovascular    0.008541
dtype: float64

`suppliers_in_shortage_list` (how many other manufacturers of the same generic drug are *also* currently listed as short) dominates feature importance. That's an intuitive, non-tautological finding: a drug with few or no alternative suppliers also on the shortage list is structurally more fragile — one company's decision to discontinue effectively is the market's supply for that product — while a drug with several suppliers still active has more room to recover as "Current" rather than being written off.

## Descriptive: discontinuation rate by category

In [6]:
discontinuation_rate_by(df, "therapeutic_category")

,discontinuation_rate,n_rows
therapeutic_category,,
Transplant,1.000000,11
Antiviral,1.000000,15
Renal,1.000000,25
Medical Imaging,0.800000,15
Anti-Infective,0.546392,97
Gastroenterology,0.500000,50
Hematology,0.500000,24
Oncology,0.462366,93
Cardiovascular,0.422360,161


In [7]:
discontinuation_rate_by(df, "dosage_form")

,discontinuation_rate,n_rows
dosage_form,,
"Capsule, Extended Release",1.000000,10
Unknown,1.000000,19
Solution,1.000000,31
Suppository,1.000000,12
Oral Suspension,0.636364,11
Tablet,0.480122,327
Capsule,0.222222,135
Injection,0.160696,977
"Tablet, Chewable",0.052632,19


Anesthesia and psychiatry drugs are rarely permanently discontinued once short (~5%); transplant, antiviral, and renal drugs are far more likely to be (near or at 100%, though on smaller sample sizes -- 11-25 rows each, so treat those as directional rather than firm). Injectable dosage forms are much less likely to be discontinued (16%) than tablets (48%) or solutions (100%, again a small-sample category).

## Summary

- Naive AUC (0.986) was inflated by train/test combination overlap in a small categorical space -- not a usable number.
- Honest, combo-grouped AUC: **~0.93 (std 0.06)** -- a real, moderately strong, generalizable signal.
- Primary driver: **supplier concentration** (how many other suppliers of the same drug are also short) -- few alternative suppliers correlates strongly with permanent discontinuation.
- Secondary, descriptive pattern: therapeutic category and dosage form both show wide, sensible variation in discontinuation rate (anesthesia/psychiatry low; transplant/antiviral/renal high; injectables lower than tablets/solutions).
- Caveat carried over from the README: this is about drugs *already on* the shortage list, not a general shortage-prediction model, and several of the higher-risk categories rest on small sample sizes.